In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import SiglipVisionModel, AutoTokenizer, AutoImageProcessor
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
from tqdm import tqdm
import numpy as np
from PIL import Image

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
def siglip_loss(image_embeddings, text_embeddings, temperature=0.07):
    image_embeddings = F.normalize(image_embeddings, dim=-1)
    text_embeddings = F.normalize(text_embeddings, dim=-1)

    logits = image_embeddings @ text_embeddings.T  # Cosine similarity
    logits = logits / temperature

    batch_size = logits.size(0)
    targets = torch.eye(batch_size).to(logits.device)  # Identity for positive pairs

    return F.binary_cross_entropy_with_logits(logits, targets)

In [4]:
class LinearProjection(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        return self.linear(x)

In [5]:
def get_text_embedding(text, tokenizer, device, max_length=256):
    if not text or len(text.strip()) == 0:
        text = "This is a placeholder description."

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=max_length
    )

    inputs = {k: v.to(device).float() for k, v in inputs.items()}
    return inputs['input_ids'].float()

In [6]:
# Load SigLIP model and tokenizer
siglip_model = SiglipVisionModel.from_pretrained("google/siglip-so400m-patch14-384")
siglip_processor = AutoImageProcessor.from_pretrained("google/siglip-so400m-patch14-384")
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct")

# Ensure pad_token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Freeze vision model
for param in siglip_model.parameters():
    param.requires_grad = False

siglip_model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


SiglipVisionModel(
  (vision_model): SiglipVisionTransformer(
    (embeddings): SiglipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
      (position_embedding): Embedding(729, 1152)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-26): 27 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): PytorchGELUTanh()
            (fc1): Linear(in_features=1152, out_fe

In [7]:
# Dummy image to get vision model output dim
dummy_image = Image.new('RGB', (384, 384), color='black')
with torch.no_grad():
    siglip_inputs = siglip_processor(dummy_image, return_tensors="pt").to(device)
    siglip_outputs = siglip_model(**siglip_inputs)
    siglip_output_dim = siglip_outputs.pooler_output.shape[-1]

# Dummy text to get tokenizer embedding dim
dummy_text = "This is a test."
dummy_embedding = get_text_embedding(dummy_text, tokenizer, device)
text_embedding_dim = dummy_embedding.shape[-1]

print(f"SigLIP output dimension: {siglip_output_dim}")
print(f"Text embedding dimension: {text_embedding_dim}")


SigLIP output dimension: 1152
Text embedding dimension: 256


In [8]:
linear_proj = LinearProjection(siglip_output_dim, text_embedding_dim).to(device)

In [9]:
from datasets import load_dataset
from torchvision.transforms import Compose, Resize, ToTensor

# Load VQA-CIFAR10 from Hugging Face
ds = load_dataset("TharunSivamani/vqa-cifar10")

dataset = ds.shuffle(seed=42)

In [10]:
import numpy as np

# Calculate the length of each answer
answer_lengths = [len(sample['answer']) for sample in dataset['train']]

# Calculate the mean and standard deviation
mean_length = np.mean(answer_lengths)
std_length = np.std(answer_lengths)

max_len = max(len(sample['answer']) for sample in dataset['train'])

# Print results
print(f"Mean answer length: {mean_length:.2f} characters")
print(f"Standard deviation of answer lengths: {std_length:.2f} characters")
print(f"Maximum answer length (in characters): {max_len}")

# Mean answer length: 114.54 characters
# Standard deviation of answer lengths: 108.80 characters
# Maximum answer length (in characters): 374

Mean answer length: 114.54 characters
Standard deviation of answer lengths: 108.80 characters
Maximum answer length (in characters): 374


In [11]:
num_images = dataset['train'].num_rows
num_images

550000

In [12]:
from torch.utils.data import Dataset

# Define image transform
transform = Compose([
    Resize((384, 384)),
    ToTensor()
])

class VQACIFAR10Dataset(Dataset):
    def __init__(self, hf_dataset, transform=None, max_samples=None):
        self.dataset = hf_dataset
        self.transform = transform
        self.max_samples = max_samples or len(hf_dataset)
        
    def __len__(self):
        return min(self.max_samples, len(self.dataset))
    
    def __getitem__(self, idx):
        data = self.dataset[idx]
        image = data['image']
        if self.transform:
            image = self.transform(image)
        text = str(data['answer'])  # use answer as the aligned text
        return image, text


In [13]:
num_images = 100  # Can change this
batch_size = 8
learning_rate = 3e-4

dummy_linear_proj = LinearProjection(siglip_output_dim, text_embedding_dim).to(device)
vqa_dataset = VQACIFAR10Dataset(dataset['train'], transform=transform, max_samples=num_images)
dummy_dataloader = DataLoader(vqa_dataset, batch_size=batch_size, shuffle=True)
dummy_optimizer = AdamW(dummy_linear_proj.parameters(), lr=learning_rate)

In [14]:
# Run only 1 step of training to check everything is working
dummy_linear_proj.train()
images, answers = next(iter(dummy_dataloader))
images = images.to(device)

print(f"Image batch shape: {images.shape}")

# Get image embeddings
with torch.no_grad():
    siglip_inputs = siglip_processor(images, return_tensors="pt", do_rescale=False).to(device)
    image_features = siglip_model(**siglip_inputs).pooler_output

print(f"Image features shape (from SigLIP): {image_features.shape}")

# Project to text embedding space
projected_features = dummy_linear_proj(image_features)
print(f"Projected image features shape: {projected_features.shape}")

# Get embeddings for all answers in the batch
text_embeddings_list = []
for i, answer in enumerate(answers):
    print(f"Answer {i}: {answer}")
    emb = get_text_embedding(answer, tokenizer, device)
    print(f"Embedding shape: {emb.shape}")
    text_embeddings_list.append(emb)

text_embeddings = torch.stack(text_embeddings_list, dim=0).squeeze(1)
print(f"Stacked text embeddings shape: {text_embeddings.shape}")

# Compute contrastive loss
loss = siglip_loss(projected_features, text_embeddings)
print(f"Loss: {loss.item()}")

# Optional: backpropagation step
dummy_optimizer.zero_grad()
loss.backward()
dummy_optimizer.step()
print("One optimization step completed.")

Image batch shape: torch.Size([8, 3, 384, 384])
Image features shape (from SigLIP): torch.Size([8, 1152])
Projected image features shape: torch.Size([8, 256])
Answer 0: The image shows a white airplane with black wings and tail.
Embedding shape: torch.Size([1, 256])
Answer 1: Yes
Embedding shape: torch.Size([1, 256])
Answer 2: This resembles a bird.
Embedding shape: torch.Size([1, 256])
Answer 3: The image gives a sense of calmness and serenity. The clear blue sky and the calm water suggest a peaceful day at sea. The solitary boat adds a touch of solitude and tranquility to the scene. The overall vibe is one of relaxation and enjoyment of nature.
Embedding shape: torch.Size([1, 256])
Answer 4: The image gives a sense of calmness and serenity. The presence of the horse, a symbol of strength and grace, in a peaceful setting with a clear sky and lush greenery, evokes feelings of tranquility and harmony. The image also conveys a sense of freedom and the beauty of nature.
Embedding shape: t

In [15]:
num_images = 50000  # Can change this
batch_size = 1024
num_epochs = 1
learning_rate = 3e-4
load_checkpoint = False
checkpoint_path = 'linear_projection.pth'

vqa_dataset = VQACIFAR10Dataset(dataset['train'], transform=transform, max_samples=num_images)
dataloader = DataLoader(vqa_dataset, batch_size=batch_size, shuffle=True)

In [16]:
optimizer = AdamW(linear_proj.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    total_loss = 0
    linear_proj.train()

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for images, answers in progress_bar:
        images = images.to(device)

        # Get image embeddings
        with torch.no_grad():
            siglip_inputs = siglip_processor(images, return_tensors="pt", do_rescale=False).to(device)
            image_features = siglip_model(**siglip_inputs).pooler_output

        # Project to text embedding space
        projected_features = linear_proj(image_features)

        # Get embeddings for all answers in the batch
        text_embeddings_list = []
        for answer in answers:
            emb = get_text_embedding(answer, tokenizer, device)
            text_embeddings_list.append(emb)

        text_embeddings = torch.stack(text_embeddings_list, dim=0).squeeze(1)

        # Compute loss
        loss = siglip_loss(projected_features, text_embeddings)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_epoch_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_epoch_loss:.4f}")


Epoch 1/1: 100%|██████████| 49/49 [16:43<00:00, 20.48s/it, loss=0.0139] 

Epoch 1/1, Average Loss: 0.0404


In [17]:
# Save the linear projection model
torch.save(linear_proj.state_dict(), "linear_projection.pth")
print("Saved linear projection model to 'linear_projection.pth'")

Saved linear projection model to 'linear_projection.pth'
